In [1]:
import getpass
import os
import re


os.environ["OPENAI_API_KEY"] = 

os.environ["LANGCHAIN_TRACING_V2"] = "true"


In [2]:

os.environ["LANGCHAIN_API_KEY"] =

os.environ["HUGGINGFACEHUB_API_TOKEN"] =

In [4]:
import bs4
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
import pandas as pd

In [5]:
df = pd.read_csv("note_book_data/CISS-Spark!.csv")

In [6]:
# drop row if 法宝引证码 is null
df = df.dropna(subset=["法宝引证码"])
# drop column if all columns are null
df = df.dropna(axis=1, how="all")

In [7]:
extraction_features = ["文件具体政策内容","制定单位/机关","行政级别","一般政策内容字段","对政策执行过程有规定","建议权宜处理字段","原始文件的文本内容 (entire original document text, only for references, not included in the finalized dataset)"]

In [8]:
df_subset = df[extraction_features]

In [9]:
df_subset.columns

Index(['文件具体政策内容', '制定单位/机关', '行政级别', '一般政策内容字段', '对政策执行过程有规定', '建议权宜处理字段',
       '原始文件的文本内容 (entire original document text, only for references, not included in the finalized dataset)'],
      dtype='object')

In [10]:
system_prompts = """
使用以下信息来回答最后的问题。你所有的回答都应该直接返回原文内容，如果你不知道答案，就说你不知道，不要试图编造答案。在回答的最后总是说"感谢您的提问"

{context}

问题: {question}
"""

hints = {
    "文件具体政策内容":"""提示：主要政策是对政策内容的高度概括，一般在文本最开头""",
    "制定单位/机关":"""提示：文件具体政策内容,制定单位/机关,行政级别一般会一起出现，其表达形式为：时间（optianl）+ 行政级别 + 制定单位/机关 + 文件具体政策内容""",
    "行政级别":"""提示：文件具体政策内容,制定单位/机关,行政级别一般会一起出现，其表达形式为：时间（optianl）+ 行政级别 + 制定单位/机关 + 文件具体政策内容""",
    "一般政策内容字段":"""提示：一般政策内容只描述政策目的，而不包含任政策本身细节。可能含有以下语句：为了，贯彻落实, 按照xx要求，依据xx规定，根据，等类似描述。除此之外，一般政策内容还可能提到国家最高领导人或机构（如"胡", "温", "习", "李", "党中央", "国务院"), 总则, 指导思想, 基本原则, 等类似概括性语句""",
    "对政策执行过程有规定":"""提示：政策执行过程一般在讨论具体的政策执行流程，可能含有以下关键词：“流程”、“程序”、“执行”、“分工”、“以/用/通过...方式/方法”、“牵头”、“由...(主要)负责”、“责任”、“措施”、“...个阶段”、“进度”、“时间进度”。""",
    "建议权宜处理字段":"""提示：权宜处理指政策可以根据实际情况进行合理变通。""",
}
examples = {
    "文件具体政策内容":"""示例1：环卫行业信用管理暂行办法；示例2:生猪养殖用地保障工作""",
    "制定单位/机关":'''示例1：广东省住房和城乡建设厅。出处：广东省住房和城乡建设厅关于印发环卫行业信用管理暂行办法的通知''',
    "行政级别":"""示例：省级。出处：广东省住房和城乡建设厅关于印发环卫行业信用管理暂行办法的通知""",
    "一般政策内容字段":'''示例：为认真贯彻落国家和省关于房地产工作的决策部署，加强和改进住房及用地供应管理，规范市场秩序，稳定市场预期，促进全市房地产市场平稳健康发展，按照适度、精准的调控原则，经市政府研究，决定对房地产市场有关政策进行调整完善。现就有关事宜通知如下：''',
    "对政策执行过程有规定":'''''',
    "建议权宜处理字段": """示例1：结合广东省环卫行业实际，制定本办法。。示例2：生猪养殖可因地制宜合理布局，"""
}

questions = {
    "文件具体政策内容":"文本主要描述的政策（政策内容）是什么，你只需返回原文内容。",
    "制定单位/机关":"文本中政策的制定单位/机关是谁，请返回原文内容，若没有相关内容请返回无",
    "行政级别":"，文本中制定政策的单位/机关的行政级别是什么，返回原文内容，若没有相关内容请返回无",
    "一般政策内容字段":"文本中哪些语句在描述一般政策内容，请返回原文所有文字",
    "对政策执行过程有规定":"文本中是否讨论了政策执行过程，请返回是或者否",
    "建议权宜处理字段":"文本主要描述权宜处理的语句有哪些，请返回原文所有文字，若没有相关内容请直接返回无",
}


In [11]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def get_answer(docs, system_prompt, hint, example, question, feature_name):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=512, chunk_overlap=20, length_function=len, is_separator_regex=False,)
    splits = text_splitter.split_text(docs)
    if feature_name == "文件具体政策内容":
        # only use the first split
        splits = [splits[0]]

    vectorstore = Chroma.from_texts(splits, embedding=OpenAIEmbeddings())
    Chroma.delete_collection(vectorstore)
    vectorstore = Chroma.from_texts(splits, embedding=OpenAIEmbeddings())
    
    retriever = vectorstore.as_retriever(search_kwargs={"k": 2})
    llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0)
    
    template = system_prompt + hint + f"\n{example}" + f"\nHelpful Answer:"
    
    custom_rag_prompt = PromptTemplate.from_template(template)
    
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | custom_rag_prompt
        | llm
        | StrOutputParser()
    )
    
    return rag_chain.invoke(question)

In [12]:
for i in range(0,len(df_subset)):
    features = "建议权宜处理字段"
    docs = df.iloc[i]["原始文件的文本内容 (entire original document text, only for references, not included in the finalized dataset)"]
    hint = hints[features]
    example = examples[features]
    question = questions[features]
    result = get_answer(docs, system_prompts, hint, example, question,features)
    # remove the "感谢您的提问" and "主要政策是" from the result by using regular expression
    result = re.sub(r"感谢您的提问。", "", result)
    result = re.sub(r"感谢您的提问！", "", result)
    result = re.sub(r"感谢您的提问", "", result)
    result = re.sub(r"主要政策是", "", result)
    # remove empty space before and after the text
    result = re.sub(r"^\s+|\s+$", "", result)
    # remove the empty line
    result = re.sub(r"\n\n", "\n", result)
    print(result)
    print("========================")

结合广东省环卫行业实际，制定本办法。
无
无
无
无


Number of requested results 2 is greater than number of elements in index 1, updating n_results = 1


上述通知印发前，使用了信用分类优选随机合理低价评定标办法，并已发布招标公告尚未开标的项目，招标人可根据实际情况决定是否暂停使用该办法。如暂停使用，须按照《广东省实施〈中华人民共和国招标投标法〉办法》（2018年修订）执行。
无
无
无
权宜处理的语句有哪些，请返回原文所有文字，若没有相关内容请直接返回无
无
无
无
无


In [60]:
from thefuzz import process
import logging
logging.getLogger().setLevel(logging.ERROR)

def find_权宜处理_regular_expr(docs):
    keywords = ["结合.*实际", "根据.*实际", "根据实际情况", "结合实际情况", "权宜", "结合本地实际", "根据本地实际","因地制宜"]
    pattern = re.compile('|'.join(keywords))
    paragraphs = re.split(r'。|\n', docs.strip())
    
    
    keywords_fuzzy = keywords = ["结合实际", "根据实际", "根据实际情况", "结合实际情况", "权宜", "结合本地实际", "根据本地实际","因地制宜"]
    
    matched_paragraphs = []
    for paragraph in paragraphs:
        if pattern.search(paragraph):
            matched_paragraphs.append(paragraph.strip())
        else:
            best_match = process.extractOne(paragraph, keywords_fuzzy)
            if best_match[1] > 70:  
                matched_paragraphs.append(paragraph.strip())

    return matched_paragraphs

for i in range(0,len(df_subset)):
    docs = df.iloc[i]["原始文件的文本内容 (entire original document text, only for references, not included in the finalized dataset)"]
    result = find_权宜处理_regular_expr(docs)
    print(result)

['为了规范广东省环卫行业秩序，建立生活垃圾处理运营单位信用体系，促进环卫行业健康、有序发展，根据《企业信息公示暂行条例》《广东省城乡生活垃圾处理条例》等相关法律法规，结合广东省环卫行业实际，制定本办法']
['在不涉及占用永久基本农田和饮用水水源保护区、自然保护地、城镇居民区、文化教育科学研究区等人口集中区域及法律法规规定的其他禁养区前提下，生猪养殖可因地制宜合理布局，允许占用一般耕地，但尽量避免占用优质耕地，特别是高标准农田']
['各区人民政府要结合全市目标和本区实际，制定本区未来3年的企业上市目标', '对企业改制上市过程中遇到的困难和涉及的相关证明手续，特别是完善用地房产手续、税收缴纳、产权厘清、人员安置等历史遗留问题，以及拟上市企业募投项目备案、核准等，市发展改革委、科技、工信、公安、司法、财政、人社、规划和自然资源、生态环境、住房城乡建设、交通运输、商务、国资、市场监管、城管、住房公积金管理等部门要根据实际情况，认真做好指导和服务，同时争取海关、外汇、税务等部门支持，依法合规加快办理', '各区人民政府根据本区实际，制定扶持政策，积极协调支持']
[]
['对农村地区燃气工程安装收费管理，可参照本指导意见并结合当地实际情况制定具体政策']
['上述通知印发前，使用了信用分类优选随机合理低价评定标办法，并已发布招标公告尚未开标的项目，招标人可根据实际情况决定是否暂停使用该办法']
['第一条\u3000为规范和加强农村“厕所革命”奖补资金管理，提高资金使用效益，根据《中华人民共和国预算法》《财政部 农业农村部关于开展农村“厕所革命”整村推进财政奖补工作的通知》《土地指标跨省域调剂收入安排的支出管理暂行办法》《广东省涉农资金统筹整合实施方案（试行）》等规定，结合农村“厕所革命”工作实际，制定本细则', '各地可根据工作实际确定具体支持内容', '第十六条\u3000各县（市、区）农业农村、财政等部门，应统筹中央和省级奖补资金、其他省级涉农资金和市县安排的补助资金，结合实际情况，科学确定本地农村“厕所革命”奖补方案，明确补助对象范围、具体补助标准、补助方式、资金管理要求等']
[]
[]
['为认真贯彻落实党中央、国务院关于安全生产领域改革发展意见，坚决防范和遏制重特大事故，推动全市安全生产形势持续稳定好转，根据青岛市安全生产工作任务总体要求，结合我市实际，

In [118]:
from thefuzz import fuzz

def find_政策执行过程_regular_expr(docs):
    # remove all empty line of docs
    docs = re.sub(r"\n\n", "\n", docs)
    # remove all content between 《》
    docs = re.sub(r"《.*》", "", docs)
    
    # 定义与政策执行过程相关的关键词
    keywords = ["(?:以|用|通过).*?方式|方法", "由.*?(?:主要负责|负责)"]
    pattern = re.compile('|'.join(keywords))
    
    # 将文档分割成段落
    paragraphs = re.split(r'。|\n', docs.strip())
    # remove ''
    paragraphs = [para for para in paragraphs if para]
    
    # 定义用于模糊搜索的关键词列表
    policy_related_phrases = ["流程", "程序", "执行", "牵头", "分工","具体执行流程", "详细分工安排", "执行阶段划分","责任", "措施", "阶段", "进度", "时间进度"]
    confirmed_matches = []
    
    for para in paragraphs[4:]:
        best_match = process.extractOne(para, policy_related_phrases)
        if best_match[1] > 70:  
            confirmed_matches.append(para.strip())
            # print(para.strip())
        if pattern.search(para):
            # print(para)
            confirmed_matches.append(para.strip())
            # print(para.strip())
    
    if confirmed_matches:
        first_pos = docs.strip().find(confirmed_matches[0])
        last_pos = docs.strip().find(confirmed_matches[-1]) + len(confirmed_matches[-1])
        # 获取首尾段落之间的文本
        relevant_text = docs.strip()[first_pos:last_pos]
        # 计算字数，排除非中文字符
        word_count = len(re.findall(r'[\u4e00-\u9fff]', relevant_text))

        return word_count
    return 0

# 假设有一个DataFrame `df_subset`，其中包含了要分析的文档
for i in range(0, len(df_subset)):
    docs = df_subset.iloc[i]["原始文件的文本内容 (entire original document text, only for references, not included in the finalized dataset)"]  # 确保列名与实际DataFrame中的列名相匹配
    result = find_政策执行过程_regular_expr(docs)
    print(result)
   
    
    

941
140
0
3905
2313
0
3153
41
1111
11
539
0
1173


In [81]:
from sentence_transformers import SentenceTransformer, util
import re
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")

def find_政策执行过程_sentence_transformer(docs):
    # 加载预训练的Sentence Transformer模型
    model = SentenceTransformer('sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2').to(device)
    
    # 将文档分割成段落
    paragraphs = re.split(r'。|\n', docs.strip())

    # 定义与政策执行过程相关的关键词
    policy_related_phrases = ["流程", "程序", "执行", "分工", "方式", "方法", "牵头", "负责", "责任", "措施", "阶段", "进度", "时间进度"]
    keywords = ["以|用|通过.*方式|方法", "由.*主要负责|负责"]
    pattern = re.compile('|'.join(keywords))

    # 将关键词列表转换为向量
    keyword_vectors = model.encode(policy_related_phrases)

    confirmed_matches = []
    for para in paragraphs:
        # 将段落转换为向量
        # detach to CPU
        para_vector = model.encode(para, convert_to_tensor=True).detach().to('cpu')
        
        # 计算段落与每个关键词向量的相似度
        similarities = util.pytorch_cos_sim(para_vector, keyword_vectors)

        # 检查是否有超过阈值的相似度得分
        if max(similarities[0]) > 0.6:  
            confirmed_matches.append(para)
        elif pattern.search(para):
            confirmed_matches.append(para.strip())

    if confirmed_matches:
        first_pos = docs.strip().find(confirmed_matches[0])
        last_pos = docs.strip().find(confirmed_matches[-1]) + len(confirmed_matches[-1])
        # 获取首尾段落之间的文本
        relevant_text = docs.strip()[first_pos:last_pos]
        # 计算字数，排除非中文字符
        word_count = len(re.findall(r'[\u4e00-\u9fff]', relevant_text))

        return word_count

# 示例文档
for i in range(0, len(df_subset)):
    docs = df_subset.iloc[i]["原始文件的文本内容 (entire original document text, only for references, not included in the finalized dataset)"]  # 确保列名与实际DataFrame中的列名相匹配
    result = find_政策执行过程_sentence_transformer(docs)
    print(result)

2748
0
0
0


KeyboardInterrupt: 